In [1]:
import math_toolkit
math_toolkit.notebook.activate()

# subs

Replace symbols, indexed labels, parameters, or whole subexpressions inside a SymPy expression while keeping the original expression unchanged.

`expr.subs(old, new)`  
`expr.subs({old1: new1, old2: new2})`  
`expr.subs([(old1, new1), ...], simultaneous=True)`


In [2]:
residual = x[t + 1] - F[t](x[t]) - b[t]
residual.subs(t, n)


-b[n] + x[n + 1] - F[n](x[n])

## Core behavior

`subs` walks a SymPy expression and returns a new expression with matching pieces replaced. The replacement target can be an atomic symbol such as `x`, a structured index such as `t`, an indexed object such as `A[t]`, or a larger expression such as `sin(x)`.

The method is structural symbolic editing. It does not mutate the original expression, choose a simplification strategy, or solve an equation for you. Use it when you already know what replacement you want to make.


## Common patterns

The one-pair form is convenient for evaluation at a point. A dictionary or list of pairs is better when a whole model receives parameter values at once.


In [3]:
polynomial = a * x**2 + b * x + c
polynomial.subs({a: 2, b: -1})


c + 2*x**2 - x

In [4]:
polynomial.subs({a: 2, b: -1, c: 3}).subs(x, 4)


31

In [5]:
oscillation = sin(x) + sin(x)**2 + cos(x)
oscillation.subs(sin(x), y)


y**2 + y + cos(x)

**Structured notation.**

Because `math_toolkit` keeps indices and function labels as real symbolic structure, substitution can relabel them directly. You do not need to rebuild names like `x_t` by hand.


In [6]:
relabelled = residual.subs(t, n)
relabelled.subs(b[n], 0)


x[n + 1] - F[n](x[n])

In [7]:
Tuple(
    F[t](x).subs(t, s),
    F(x[t]).subs(t, s),
)


(F[s](x), F(x[s]))

In [8]:
update = x[t + 1] @Eq@ (A[t] * x[t] + b[t])
update.subs(t, n)


Eq(x[n + 1], A[n]*x[n] + b[n])

In [9]:
update.subs({A[t]: a, b[t]: 0})


Eq(x[t + 1], a*x[t])

## Options and Details

**Controlling order.** Ordered pair lists are applied from left to right. That matters when replacements overlap. Use `simultaneous=True` when the pairs should describe one coordinated replacement step, such as swapping variables.

In [15]:
swap_target = x + 2 * y
Tuple(
    swap_target.subs([(x, y), (y, x)]),
    swap_target.subs([(x, y), (y, x)], simultaneous=True),
)


(3*x, 2*x + y)

In [16]:
ratio = x / y
Tuple(
    ratio.subs([(x, y), (y, x)]),
    ratio.subs([(x, y), (y, x)], simultaneous=True),
)


(1, y/x)

In [17]:
shift = x + 1
once = shift.subs(x, x + 1)
twice = once.subs(x, x + 1)
Tuple(once, twice)


(x + 2, x + 3)

## Semantics

`subs` respects the expression you give it. If the shape you want is hidden by expansion, factor or rewrite first. If the question is "what value of `x` makes this true?", substitute known values first and then call a solving tool.

Bound variables inside sums and integrals are protected. Substituting a free parameter changes the integrand; substituting the integration variable itself does not rename the bound variable.


In [18]:
quadratic = x**2 + 2 * x + 1
Tuple(
    quadratic.subs(x + 1, y),
    factor(quadratic).subs(x + 1, y),
)


(x**2 + 2*x + 1, y**2)

In [19]:
equation = (x + y) @Eq@ 3
reduced = equation.subs(y, 1)
Tuple(reduced, solve(reduced, x)[0])


(Eq(x + 1, 3), 2)

In [20]:
integral = Integral(x * y, (x, 0, 1))
Tuple(
    integral.subs(x, z),
    integral.subs(y, z),
)


(Integral(x*y, (x, 0, 1)), Integral(x*z, (x, 0, 1)))

In [21]:
Tuple(
    exp(x).subs(x, 1),
    exp(x).subs(x, 1).evalf(),
)


(E, 2.71828182845905)

## Examples and Applications

### Example: sample and relabel a symbolic conditional

Substitution is the small operation behind many notebook workflows: evaluate a piecewise expression at sample values, relabel a discrete model, insert a computed state into an equation, or expand a named definition and then specialize it.

In [10]:
abs_x = If(x < 0).Then(-x).Otherwise(x)
[abs_x.subs(x, value) for value in (-3, 0, 2)]


[3, 0, 2]

In [11]:
abs_x.subs(x, y - 1)


Piecewise((1 - y, y < 1), (y - 1, True))

### Application: specialize a named potential

Expand a readable named formula at the moment when ordinary symbolic operations need access to the underlying expression.

In [12]:
@NamedFunction
def Potential(q):
    return q**4 / 4 - q**2 / 2

Potential(q).subs(q, x[t]) @Eq@ Potential(x[t])


True

In [13]:
expanded_potential = Potential(q)
expanded_potential.subs(q, x[t])


x[t]**4/4 - x[t]**2/2

In [14]:
force = -diff(expanded_potential, q)
force.subs(q, x[t])


-x[t]**3 + x[t]

## Pitfalls

Do not use `subs` as a vague "make it nicer" command. For representation changes, use `rewrite`; for algebraic cleanup, use `simplify`, `factor`, or a targeted simplifier; for unknown values, use `solve` or another equation-solving routine.

Be careful with overlapping replacements. When the new value from one pair contains the old value from a later pair, sequential substitution can cascade. Reach for `simultaneous=True` when the replacements are conceptually one step.


## See also

[Expression](Expression.ipynb)  
[Indexed](Indexed.ipynb)  
[Function](Function.ipynb)  
[If](If.ipynb)  
[rewrite](rewrite.ipynb)  
[Composing structured expressions](../tutorials/composing_expressions.ipynb)  
[Indexing notation](../tutorials/indexing.ipynb)  
[Function families](../tutorials/function_families.ipynb)
